## 1. Connect to PostgreSQL and set up the embedding service
Loads the database URL and the Jina key from `.env` (never printed), opens the database connection, and sets the embedding model and vector size (768, matching the `chunks` table).

In [8]:
# --- Imports ---
import os, time, random
import numpy as np
import pandas as pd
import psycopg
import requests
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

# --- Load secrets from .env (override=True picks up edits made after the kernel started) ---
load_dotenv(os.path.expanduser("~/bank-rag-guard/.env"), override=True)

# --- Database connection (autocommit saves each statement immediately) ---
conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)
register_vector(conn)

# --- Settings ---
DATA_DIR = os.path.expanduser("~/bank-rag-guard/data")
EMBED_URL = "https://api.jina.ai/v1/embeddings"    # Jina's embedding endpoint
EMBED_MODEL = "jina-embeddings-v3"                 # multilingual (includes Arabic)
DIM = 768                                          # must match vector(768) in the chunks table
JINA_HEADERS = {"Authorization": f"Bearer {os.environ['JINA_API_KEY']}",   # key sent in a header,
                "Content-Type": "application/json"}                         # never printed
random.seed(42)                                    # reproducible tier assignment
print("Connected")

Connected


## 2. Test the embedding service
Embeds one word and checks the vector length. **Expected output:** `HTTP 200` and `Vector length: 768`.

In [9]:
# --- Embed one word with the same settings the real pipeline will use ---
r = requests.post(EMBED_URL, headers=JINA_HEADERS, timeout=30, json={
    "model": EMBED_MODEL,
    "task": "retrieval.passage",     # tells the model this is stored document text
    "dimensions": DIM,               # ask for 768-number vectors
    "input": ["hello"],
})
print("HTTP", r.status_code)
if r.status_code == 200:
    print("Vector length:", len(r.json()["data"][0]["embedding"]))
else:
    print(r.text[:300])              # Google-style error text; contains no key

HTTP 200
Vector length: 768


## 3. Embedding helper and Arabic/English sanity check
`embed_texts` embeds a list of texts and retries with growing waits if the service says "too many requests" (HTTP 429) or is briefly down. Vectors are normalised to length 1 for cosine search.
The check embeds an English sentence, the same sentence in Arabic, and an unrelated sentence. **Expected result:** the English/Arabic similarity is clearly higher than English/unrelated.

In [10]:
def embed_texts(texts, task_type="RETRIEVAL_DOCUMENT", max_retries=6):
    """Embed a list of strings; returns a numpy array of shape (len(texts), DIM)."""
    # Translate our task names into Jina's task names
    task = "retrieval.query" if task_type == "RETRIEVAL_QUERY" else "retrieval.passage"
    for attempt in range(max_retries):
        resp = requests.post(EMBED_URL, headers=JINA_HEADERS, timeout=60, json={
            "model": EMBED_MODEL, "task": task, "dimensions": DIM, "input": texts})
        if resp.status_code == 200:
            data = sorted(resp.json()["data"], key=lambda d: d["index"])   # keep input order
            vecs = np.array([d["embedding"] for d in data], dtype=np.float32)
            vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)           # normalise each vector
            return vecs
        if resp.status_code in (429, 500, 502, 503):      # busy or rate limited: wait and retry
            wait = 5 * 2 ** attempt                        # 5s, 10s, 20s, 40s, ...
            print(f"  HTTP {resp.status_code}, retrying in {wait}s")
            time.sleep(wait)
            continue
        raise RuntimeError(f"Embedding failed: HTTP {resp.status_code} {resp.text[:200]}")
    raise RuntimeError("Embedding failed after several retries")

# --- Bilingual sanity check: same meaning in two languages vs an unrelated sentence ---
sentences = [
    "The customer reported a fraudulent card transaction",   # English
    "أبلغ العميل عن عملية احتيال بالبطاقة",                    # same meaning in Arabic
    "The weather is sunny and warm today",                   # unrelated
]
v = embed_texts(sentences)
sim = v @ v.T        # for normalised vectors, dot product = cosine similarity
print("English vs Arabic   :", round(float(sim[0, 1]), 3))
print("English vs unrelated:", round(float(sim[0, 2]), 3))

English vs Arabic   : 0.784
English vs unrelated: 0.341


## Check which data files exist
Lists the `data/` folder. Cell 5 needs `customers.csv`, `enron_emails.jsonl` and `cuad_contracts.jsonl`.

In [11]:
# --- List every file in the data folder with its size ---
for name in sorted(os.listdir(DATA_DIR)):
    size_kb = os.path.getsize(os.path.join(DATA_DIR, name)) / 1024
    print(f"{name:40s} {size_kb:10.1f} KB")

cuad_contracts.jsonl                        26492.6 KB
customers.csv                                  93.4 KB
enron_emails.jsonl                           2705.3 KB
pii_benchmark.jsonl                          1483.4 KB
prompt_injections_test.csv                     14.7 KB
prompt_injections_train.csv                    65.6 KB


## 4. Prepare chunks and assign clearance tiers
Combines three sources into one list of chunks:
- **customer_notes** (300): keep the tier generated earlier
- **enron** (600 emails): random tier, 60% tier 1 / 30% tier 2 / 10% tier 3
- **cuad** (40 contracts, 10 chunks each): contracts are more sensitive, so tier 2 (60%) or tier 3 (40%)

Tiers are synthetic labels used to test access control. Each contract keeps one tier for all its chunks.

In [12]:
# --- Sizes: kept small to stay inside free embedding quotas ---
N_EMAILS, N_CONTRACTS, CHUNKS_PER_CONTRACT = 600, 40, 10

def chunk_text(text, size=1000, overlap=100):
    """Split text into pieces of `size` characters, overlapping by `overlap`."""
    pieces, start = [], 0
    while start < len(text):
        pieces.append(text[start:start + size])
        start += size - overlap
    return pieces

records = []   # every chunk becomes one dict with source, doc_id, content, tier

# --- 1) Customer notes (tier already assigned in notebook 02) ---
customers = pd.read_csv(f"{DATA_DIR}/customers.csv")
for r in customers.itertuples():
    records.append({"source": "customer_notes", "doc_id": r.customer_id,
                    "content": r.text, "tier": int(r.tier)})

# --- 2) Enron emails: one chunk each, trimmed to 1,500 characters ---
emails = pd.read_json(f"{DATA_DIR}/enron_emails.jsonl", lines=True,
                      dtype=False, convert_dates=False).head(N_EMAILS)
for i, text in enumerate(emails["text"]):
    tier = random.choices([1, 2, 3], weights=[60, 30, 10])[0]
    records.append({"source": "enron", "doc_id": f"enron-{i:04d}",
                    "content": text[:1500], "tier": tier})

# --- 3) CUAD contracts: first few chunks of each, all sharing one tier ---
contracts = pd.read_json(f"{DATA_DIR}/cuad_contracts.jsonl", lines=True,
                         dtype=False, convert_dates=False).head(N_CONTRACTS)
for c in contracts.itertuples():
    tier = random.choices([2, 3], weights=[60, 40])[0]
    for j, piece in enumerate(chunk_text(c.text)[:CHUNKS_PER_CONTRACT]):
        records.append({"source": "cuad", "doc_id": f"{c.title}#{j}",
                        "content": piece, "tier": tier})

# --- Summary: chunks per source and tier ---
summary = pd.DataFrame(records).groupby(["source", "tier"]).size().unstack(fill_value=0)
print(len(records), "chunks in total")
summary

1284 chunks in total


tier,1,2,3
source,,,
cuad,0,282,102
customer_notes,148,100,52
enron,351,190,59


## 5. Embed and load into `chunks`
Embeds the chunks in small batches, pausing between batches to respect rate limits, and inserts each batch immediately. **It is safe to stop and re-run:** chunks already in the table are skipped. Expect roughly 10-15 minutes.

In [13]:
import sys
sys.path.insert(0, os.path.expanduser("~/bank-rag-guard/src"))
import masker                                          # same masking as the answer pipeline

BATCH_SIZE = 16        # texts per API call
PAUSE_SECONDS = 2      # wait between calls (lower it if your quota is generous)

# --- Skip anything already loaded, so re-running resumes where it stopped ---
existing = {row[0] for row in conn.execute("SELECT doc_id FROM chunks").fetchall()}
todo = [r for r in records if r["doc_id"] not in existing]
print(f"{len(existing)} already loaded, {len(todo)} to do")

for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i + BATCH_SIZE]
    # Embed the MASKED text, so no raw personal data is sent to the embedding service.
    # The raw text is still stored below: retrieval and the tier rule run inside the database.
    vecs = embed_texts([masker.mask_all(r["content"])[0] for r in batch])      # one API call per batch

    for r, v in zip(batch, vecs):                          # save this batch right away
        conn.execute(
            "INSERT INTO chunks (source, doc_id, content, tier, embedding) "
            "VALUES (%s, %s, %s, %s, %s)",
            (r["source"], r["doc_id"], r["content"], r["tier"], v))

    print(f"inserted {min(i + BATCH_SIZE, len(todo))}/{len(todo)}")
    time.sleep(PAUSE_SECONDS)                              # stay under the rate limit

0 already loaded, 1284 to do
inserted 16/1284
inserted 32/1284
inserted 48/1284
inserted 64/1284
inserted 80/1284
inserted 96/1284
inserted 112/1284
inserted 128/1284
inserted 144/1284
inserted 160/1284
inserted 176/1284
inserted 192/1284
inserted 208/1284
inserted 224/1284
inserted 240/1284
inserted 256/1284
inserted 272/1284
inserted 288/1284
inserted 304/1284
inserted 320/1284
inserted 336/1284
inserted 352/1284
inserted 368/1284
inserted 384/1284
inserted 400/1284
inserted 416/1284
inserted 432/1284
inserted 448/1284
inserted 464/1284
inserted 480/1284
inserted 496/1284
inserted 512/1284
inserted 528/1284
inserted 544/1284
inserted 560/1284
inserted 576/1284
inserted 592/1284
inserted 608/1284
inserted 624/1284
inserted 640/1284
inserted 656/1284
inserted 672/1284
inserted 688/1284
inserted 704/1284
inserted 720/1284
inserted 736/1284
inserted 752/1284
inserted 768/1284
inserted 784/1284
inserted 800/1284
inserted 816/1284
inserted 832/1284
inserted 848/1284
inserted 864/1284
inser

## 6. Verify what was loaded
Counts chunks per source and tier. The numbers should match the summary from Cell 5.

In [14]:
rows = conn.execute("""
    SELECT source, tier, COUNT(*) FROM chunks
    GROUP BY source, tier ORDER BY source, tier
""").fetchall()
pd.DataFrame(rows, columns=["source", "tier", "chunks"])

,source,tier,chunks
0,cuad,2,282
1,cuad,3,102
2,customer_notes,1,148
3,customer_notes,2,100
4,customer_notes,3,52
5,enron,1,351
6,enron,2,190
7,enron,3,59


## 7. Tier-leakage test: restricted chunks never reach a lower tier
`retrieve` embeds a question and returns the closest chunks, filtered with `WHERE tier <= user_tier` **inside the database query**. The test runs four questions as tier 1, 2 and 3 users, twice each:
- **with the filter** (the real system): `leaks_with_filter` must be 0
- **without the filter** (a control): `would_leak_without_filter` shows how many restricted chunks the filter blocked

The assertion fails loudly if anything leaks.

In [15]:
from functools import lru_cache

@lru_cache(maxsize=None)
def embed_query(q):
    """Embed a search query once and remember the result."""
    return embed_texts([q], task_type="RETRIEVAL_QUERY")[0]

def retrieve(query, user_tier, k=5, enforce=True):
    """Return the k closest chunks. With enforce=True, only chunks the user may see."""
    q = embed_query(query)
    where = "WHERE tier <= %s " if enforce else ""      # the access-control filter
    sql = (f"SELECT source, doc_id, tier, content FROM chunks {where}"
           "ORDER BY embedding <=> %s LIMIT %s")        # <=> is cosine distance
    params = (user_tier, q, k) if enforce else (q, k)
    return conn.execute(sql, params).fetchall()

# --- Questions chosen to hit sensitive material ---
test_queries = [
    "disputed card transaction",
    "contract termination and liability clause",
    "meeting schedule and trading positions",
    "customer Emirates ID and IBAN",
]

report = []
for tier in (1, 2, 3):
    for q in test_queries:
        secured = retrieve(q, tier, k=10, enforce=True)      # real behaviour
        control = retrieve(q, tier, k=10, enforce=False)     # what an unprotected system returns
        report.append({
            "user_tier": tier,
            "query": q,
            "leaks_with_filter": sum(1 for r in secured if r[2] > tier),
            "would_leak_without_filter": sum(1 for r in control if r[2] > tier),
        })

report_df = pd.DataFrame(report)

# --- Fail loudly if any restricted chunk got through ---
assert report_df["leaks_with_filter"].sum() == 0, "TIER LEAK DETECTED"
print("PASS: zero leaks with the filter enabled")
report_df

PASS: zero leaks with the filter enabled


,user_tier,query,leaks_with_filter,would_leak_without_filter
0,1,disputed card transaction,0,6
1,1,contract termination and liability clause,0,10
2,1,meeting schedule and trading positions,0,6
3,1,customer Emirates ID and IBAN,0,5
4,2,disputed card transaction,0,1
5,2,contract termination and liability clause,0,3
6,2,meeting schedule and trading positions,0,0
7,2,customer Emirates ID and IBAN,0,1
8,3,disputed card transaction,0,0
9,3,contract termination and liability clause,0,0
